# Stage 5: Common Normalisation

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb             
-----------------------------------------------------------------            
│   ├─ Stage_5_Common_Normalisation.ipynb                       │
-----------------------------------------------------------------
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

### To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect pandas pillow praw requests import-ipynb contractions pyspellchecker nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=a92964f22229f066e7b20eae842dd82199089fbaf3d8db215f7dac5b043ea40f
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
import os

FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import re
import import_ipynb
import pandas as pd
from tqdm import tqdm
from typing import Any
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, save_single_df_to_csv, output_base_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Expand Shortforms and Slang
Address shortforms and internet slang by using a dictionary-based approach or custom mapping to expand common abbreviations (e.g., 'lol' to 'laughing out loud', 'brb' to 'be right back') and replace slang terms with their more formal equivalents.
slang_dictionary includes slangs, shortforms and emoticons.


#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### Unit Test

In [ ]:
# Load slang dictionary
slang_df = pd.read_csv('./utilities/slang_dictionary.csv', encoding='utf-8')
slang_dict = slang_df.set_index('slang')['expand'].to_dict()

# Compile one regex pattern for all slang words
pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in slang_dict.keys()) + r')\b', flags=re.IGNORECASE)

# Vectorized function for pandas Series
def expand_slang_vectorized(text_series):
    text_series = text_series.fillna("").astype(str)
    return text_series.str.replace(pattern, lambda m: slang_dict[m.group(0).lower()], regex=True)

# Example usage
example_series = pd.Series(["tmr got work alr that's cool", "idk what to do :("])
expanded_series = expand_slang_vectorized(example_series)
print(expanded_series)

0    tomorrow got work already that's cool
1              i do not know what to do :(
dtype: object


#### Shortforms Expansion

In [ ]:
# Load slang dictionary
slang_df = pd.read_csv('./utilities/slang_dictionary.csv', encoding='utf-8')
slang_dict = slang_df.set_index('slang')['expand'].to_dict()
print(f"Loaded slang dictionary with {len(slang_dict)} entries.")

# Compile regex pattern for all slang words
pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in slang_dict.keys()) + r')\b', flags=re.IGNORECASE)

# Vectorized slang expansion function
def expand_slang_vectorized(text_series):
    text_series = text_series.fillna("").astype(str)
    return text_series.str.replace(
        pattern,
        lambda m: slang_dict.get(m.group(0).lower(), m.group(0)),
        regex=True
    )

# Expand slang for each DataFrame
def expand_slang_in_df(df, text_columns):
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            print(f"Expanding slang in column '{col}'...")

            new_col_name = f"expanded_{col.replace('english_converted_', '')}"

            df[new_col_name] = expand_slang_vectorized(df[col])

    return df

Loaded slang dictionary with 5557 entries.


#### PostVault

In [ ]:
text_columns = ['english_converted_title', 'english_converted_content']

df_posts = expand_slang_in_df(df_posts, text_columns)

Expanding slang in column 'english_converted_title'...
Expanding slang in column 'english_converted_content'...


#### CommentVault

In [ ]:
text_columns = ['english_converted_text']

df_comments = expand_slang_in_df(df_comments, text_columns)

#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,,,,,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,,,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore transit point uk bound ...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

### Handle Emojis
Process emojis by converting them into descriptive text. Using emoji.demojize()


In [ ]:
import sys
!{sys.executable} -m pip install emoji

In [ ]:
import emoji
import pandas as pd
import os

def replace_emojis_with_text(text):
    if not isinstance(text, str):
        return ""
    return emoji.demojize(text)

def process_emojis_in_df(df, text_columns):
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            print(f"Processing emojis in column '{col}'...")
            new_col_name = "demojized_title" if 'title' in col else "demojized_content"

            df[new_col_name] = df[col].apply(lambda x: str(x) if pd.notna(x) else x)

            df[new_col_name] = df[new_col_name].apply(replace_emojis_with_text)

    return df

#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### PostVault

In [ ]:
text_columns = ['english_converted_title', 'english_converted_content']

df_posts = process_emojis_in_df(df_posts, text_columns)

Processing emojis in column 'english_converted_title'...
Processing emojis in column 'english_converted_content'...


#### CommentVault

In [ ]:
text_columns = ['english_converted_text']

df_comments = process_emojis_in_df(df_comments, text_columns)

#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,,,,,,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore transit point uk bound ...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

### Spelling Correction
Perform spelling correction on the text data to fix common typos and misspellings, ensuring higher quality input for sentiment analysis.

In [ ]:
import sys
!{sys.executable} -m pip install pyspellchecker nltk

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from functools import lru_cache
from spellchecker import SpellChecker
from multiprocessing import Pool, cpu_count

# Initialize SpellChecker
spell = SpellChecker()

# Cache correction results (huge speed boost)
@lru_cache(maxsize=50000)
def cached_correct(word):
    return spell.correction(word) or word


def correct_spelling(text):
    if not isinstance(text, str):
        return ""

    words = text.split()

    # Only correct unknown words
    misspelled = spell.unknown(words)

    corrected_words = [
        cached_correct(w) if w in misspelled else w
        for w in words
    ]

    return " ".join(corrected_words)


# Parallel processor
def parallel_spellcheck(texts):
    cores = cpu_count()
    print(f"Using {cores} CPU cores...")

    with Pool(cores) as pool:
        results = list(
            tqdm(
                pool.imap(correct_spelling, texts, chunksize=100),
                total=len(texts),
                desc="Spellchecking"
            )
        )

    return results


def spellcheck_df(df, text_columns):
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            print(f"Correcting spelling in column '{col}'")

            new_col_name = "spellchecked_title" if 'title' in col else "spellchecked_content"

            df[new_col_name] = df[col].apply(correct_spelling)

    return df

#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### PostVault

In [ ]:
text_columns = ['demojized_title', 'demojized_content']

df_posts = spellcheck_df(df_posts, text_columns)

Correcting spelling in column 'demojized_title'
Correcting spelling in column 'demojized_content'


#### CommentVault

In [ ]:
text_columns = ['demojized_text']

df_comments = spellcheck_df(df_comments, text_columns)

#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,,,,,,,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,bangkok post singapore transit point uk bound ...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

### Stop Word Removal, Lemmatization
Break down the cleaned and normalized text into individual words (tokens) and then remove common stop words (e.g., 'the', 'is', 'a') that do not carry significant sentiment, to focus on more meaningful terms.

Reduce words to their base or root form (lemmatization) to group together different inflected forms of a word (e.g., 'running', 'runs', 'ran' all become 'run'). This helps in reducing the vocabulary size and improves the accuracy of sentiment analysis.


NOTE: May not need tokenize

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import pandas as pd

# Download necessary NLTK data
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', quiet=True)
print("Downloading 'stopwords' corpus...")
nltk.download('stopwords', quiet=True)
print("Downloading 'punkt_tab' resource (if available and needed)...")
nltk.download('punkt_tab', quiet=True)

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("NLTK 'wordnet' and 'omw-1.4' resources downloaded.")

NLTK 'wordnet' and 'omw-1.4' resources downloaded.


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import pandas as pd
import os

# Get the list of English stop words
stop_words = set(stopwords.words('english'))

# Remove 'not' from the stop words list as it can be important for sentiment
if 'not' in stop_words:
    stop_words.remove('not')

# Instantiate WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def stopwords_and_lemmatize(text):
    if not isinstance(text, str):
        return ""

    # Replace 'not' with 'not_' for negation handling #TODO not sure if this is best yet
    text = text.replace('not', 'not_')

    # Tokenize the text into words
    tokens = word_tokenize(text)

    # Convert to lowercase and remove stop words
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

    # Lemmatize each word
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]

    # Join the tokens back into a string
    return " ".join(lemmatized_tokens)

def process_text_df(df, text_columns):
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            print(f"Processing column '{col}'...")

            new_col_name = "lemmatized_title" if 'title' in col else "lemmatized_content"

            # ensure string
            df[new_col_name] = df[col].apply(lambda x: str(x) if pd.notna(x) else "")

            df[new_col_name] = df[new_col_name].apply(stopwords_and_lemmatize)

    return df

#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### PostVault

In [ ]:
text_columns = ['demojized_title', 'demojized_content']

df_posts = process_text_df(df_posts, text_columns)

Processing column 'demojized_title'...
Processing column 'demojized_content'...


#### CommentVault

In [ ]:
text_columns = ['demojized_text']

df_comments = process_text_df(df_comments, text_columns)

#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,hot hideout mala chain co founder 26 diagnosed...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

### Trailing Cleaning

In [ ]:
import pandas as pd
import re

TRAILING_REMOVED_RE = re.compile(r"\s*\bremoved\s*$", flags=re.IGNORECASE)

def strip_trailing_removed(text: str) -> str:
    return TRAILING_REMOVED_RE.sub("", text).rstrip()

def clean_removed(
    df: pd.DataFrame,
    content_col: str = "content",
    col_to_clean: str = "lemmatized_content",
) -> pd.DataFrame:

    if content_col not in df.columns:
        raise ValueError(f"Column '{content_col}' not found")

    if col_to_clean not in df.columns:
        raise ValueError(f"Column '{col_to_clean}' not found")

    mask = df[content_col].astype(str).str.strip() == "[removed]"
    print(f"Rows with {content_col}='[removed]': {mask.sum()}")

    result = df.copy()
    result.loc[mask, col_to_clean] = (
        result.loc[mask, col_to_clean]
        .astype(str)
        .apply(strip_trailing_removed)
    )

    changed = (
        result.loc[mask, col_to_clean] != df.loc[mask, col_to_clean]
    ).sum()
    print(f"{col_to_clean} updated in {changed} rows")

    return result

#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### PostVault

In [ ]:
print("---------")
print("PostVault")
print("---------")
print(f"Loaded {len(df_posts)} rows")
df_posts = clean_removed(df_posts, content_col="content", col_to_clean="lemmatized_title")
df_posts = clean_removed(df_posts, content_col="content", col_to_clean="lemmatized_content")

---------
PostVault
---------
Loaded 31957 rows
Rows with content='[removed]': 17856
lemmatized_title updated in 2 rows
Rows with content='[removed]': 17856
lemmatized_content updated in 17856 rows


#### CommentVault

In [ ]:
print("------------")
print("CommentVault")
print("------------")
print(f"Loaded {len(df_comments)} rows")
df_comments = clean_removed(df_comments, content_col="text", col_to_clean="lemmatized_text")


#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
display(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,hot hideout mala chain co founder 26 diagnosed...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,pair adjoining ground floor hdb shop pasir ri ...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,video call scam targeting elderly 90 year old ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,woodland checkpoint officer dragged motorist f...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,,,,,,,,bangkok post singapore transit point uk bound ...,1.0


(31957, 33)

Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")

### Title and Content Concatenation

In [ ]:
import pandas as pd
import re

def concat_title_and_content(df: pd.DataFrame, col1: str, col2: str, new_col_name: str) -> pd.DataFrame:
    df[new_col_name] = df[col1] + ' ' + df[col2]
    return df

#### Load DataFrame

In [ ]:
#  Load DataFrames
df_posts = pd.read_csv('./data/PostVault.csv', low_memory=False, keep_default_na=False)
df_comments = pd.read_csv('./data/CommentVault.csv', low_memory=False, keep_default_na=False)

#### PostVault

In [ ]:
print("---------")
print("PostVault")
print("---------")
print(f"Loaded {len(df_posts)} rows")
df_posts = concat_title_and_content(df_posts, "lemmatized_title", "lemmatized_content", "lemmatized_full_text")

---------
PostVault
---------
Loaded 31957 rows


#### CommentVault

In [ ]:
print("------------")
print("CommentVault")
print("------------")
print(f"Loaded {len(df_comments)} rows")
df_comments = concat_title_and_content(df_comments)

------------
CommentVault
------------
Loaded 764462 rows


TypeError: concat_topic_and_content() missing 3 required positional arguments: 'col1', 'col2', and 'new_col_name'

#### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
display(df_posts.shape)
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/CS5246Project/data/", "PostVault")

---------
PostVault
---------


,id,title,content,score,upvote_ratio,num_comments,created_utc,subreddit_id,has_emoji,year,...,lemmatized_title,cleaned_content,singlish_normalized_content,english_converted_content,expanded_content,demojized_content,spellchecked_content,lemmatized_content,lemmatized_full_text,tfidf_cluster
0,1qs6nd8,"A Hot Hideout mala chain co-founder, 26, diagn...",,354.0,0.95,19.0,2026-01-31 16:21:49+00:00,t5_2qh8c,False,2026.0,...,hot hideout mala chain co founder 26 diagnosed...,,,,,,,,hot hideout mala chain co founder 26 diagnosed...,6.0
1,1qs5hqp,Fantasy meets whimsy at medieval-themed renais...,,153.0,0.99,15.0,2026-01-31 15:37:29+00:00,t5_2qh8c,False,2026.0,...,fantasy meet whimsy medieval themed renaissanc...,,,,,,,,fantasy meet whimsy medieval themed renaissanc...,6.0
2,1qs39j1,Pair of adjoining ground-floor HDB shops in Pa...,the property has a remaining tenure of 63 yea...,87.0,0.89,24.0,2026-01-31 14:07:22+00:00,t5_2qh8c,False,2026.0,...,pair adjoining ground floor hdb shop pasir ri ...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,the property has a remaining tenure of 63 year...,property remaining tenure 63 year available sa...,pair adjoining ground floor hdb shop pasir ri ...,6.0
3,1qs1sek,Video call scam targeting elderly,My 90-year-old mum just got a video call from ...,182.0,0.96,17.0,2026-01-31 13:02:27+00:00,t5_2qh8c,False,2026.0,...,video call scam targeting elderly,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,my 90 year old mum just got a video call from ...,90 year old mum got video call someone full po...,video call scam targeting elderly 90 year old ...,1.0
4,1qrzbgd,Woodlands Checkpoint officer dragged by motori...,,104.0,0.92,20.0,2026-01-31 10:51:19+00:00,t5_2qh8c,False,2026.0,...,woodland checkpoint officer dragged motorist f...,,,,,,,,woodland checkpoint officer dragged motorist f...,1.0


(31957, 33)

Saving DataFrames to '/content/drive/MyDrive/CS5246Project/data/' with name 'PostVault'...
----------------------------------------------------------------------------------------------------
PostVault saved to /content/drive/MyDrive/CS5246Project/data/PostVault.csv
----------------------------------------------------------------------------------------------------


In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/CS5246Project/data/", "CommentVault")